In [3]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["OMP_NUM_THREADS"] = "1"
from mpi4py import MPI
import sys
# torch
import torch


# quimb
import quimb.tensor as qtn
import autoray as ar

from vmc_torch.experiment.tn_model import *
from vmc_torch.experiment.tn_model import init_weights_to_zero, init_weights_uniform
from vmc_torch.sampler import MetropolisExchangeSamplerSpinful, MetropolisExchangeSamplerSpinful_2D_reusable
from vmc_torch.variational_state import Variational_State
from vmc_torch.optimizer import SGD, SR, DecayScheduler
from vmc_torch.VMC import VMC
from vmc_torch.hamiltonian_torch import spinful_Fermi_Hubbard_square_lattice_torch
from vmc_torch.torch_utils import SVD,QR,QR_tao
from vmc_torch.fermion_utils import unpack_ftns
from vmc_torch.global_var import DEBUG
from vmc_torch.utils import closest_divisible

pwd = '/home/sijingdu/TNVMC/VMC_code/vmc_torch/data'
# Register safe SVD and QR functions to torch
ar.register_function('torch','linalg.svd',SVD.apply)
ar.register_function('torch','linalg.qr',QR.apply)


COMM = MPI.COMM_WORLD
SIZE = COMM.Get_size()
RANK = COMM.Get_rank()

# Hamiltonian parameters
Lx = int(4)
Ly = int(4)
symmetry = 'U1SU'
t = 1.0
U = 8.0
N_f = int(Lx*Ly-2)
n_fermions_per_spin = (N_f//2, N_f//2)
H = spinful_Fermi_Hubbard_square_lattice_torch(Lx, Ly, t, U, N_f, pbc=False, n_fermions_per_spin=n_fermions_per_spin)
graph = H.graph
# TN parameters
D = 4
chi = 16
dtype=torch.float64

# Load PEPS
appendix = ''
if symmetry == 'U1_Z2':
    symmetry = 'Z2'
    appendix = '_U1'
elif symmetry == 'U1SU':
    symmetry = 'Z2'
    appendix = '_U1SU'
    
# Load PEPS
params_path = pwd+f"/{Lx}x{Ly}/t={t}_U={U}/N={N_f}/{symmetry}/D={D}/peps_su_params{appendix}.pkl"
skeleton_path = pwd+f"/{Lx}x{Ly}/t={t}_U={U}/N={N_f}/{symmetry}/D={D}/peps_skeleton{appendix}.pkl"
peps = unpack_ftns(params_path=params_path, skeleton_path=skeleton_path, dtype=dtype, scale=1.0, new_symmray_format=True)

# VMC sample size
N_samples = int(1e3)
N_samples = closest_divisible(N_samples, SIZE)
if (N_samples/SIZE)%2 != 0:
    N_samples += SIZE

# Set up variational model
contraction_kwargs = {
    "mode": "fit",
    "bsz": 2,
    "max_iterations": 50,
    "tn_fit": "zipup",
    "progbar": False,
    "tol": 1e-5,
}
contraction_kwargs = {"mode": "mps"}
# model = fTNModel(peps, max_bond=chi, dtype=dtype)
model = fTNModel_reuse(peps, max_bond=chi, dtype=dtype, debug=False, contraction_kwargs=contraction_kwargs)

In [4]:
params, skeleton = qtn.pack(peps)

In [9]:
import quimb as qu
ftn_params, ftn_params_pytree = qu.utils.tree_flatten(params, get_ref=True)
ftn_params_pytree

{0: {(0, 1, 0): Leaf, (0, 0, 1): Leaf, (1, 0, 0): Leaf, (1, 1, 1): Leaf},
 1: {(0, 0, 1, 0): Leaf,
  (1, 1, 1, 0): Leaf,
  (0, 0, 0, 1): Leaf,
  (1, 1, 0, 1): Leaf,
  (0, 1, 0, 0): Leaf,
  (1, 0, 0, 0): Leaf,
  (0, 1, 1, 1): Leaf,
  (1, 0, 1, 1): Leaf},
 2: {(0, 0, 1, 0): Leaf,
  (1, 1, 1, 0): Leaf,
  (0, 0, 0, 1): Leaf,
  (1, 1, 0, 1): Leaf,
  (0, 1, 0, 0): Leaf,
  (1, 0, 0, 0): Leaf,
  (0, 1, 1, 1): Leaf,
  (1, 0, 1, 1): Leaf},
 3: {(0, 0, 0): Leaf, (1, 0, 1): Leaf, (0, 1, 1): Leaf, (1, 1, 0): Leaf},
 4: {(0, 0, 1, 0): Leaf,
  (1, 0, 1, 1): Leaf,
  (0, 1, 0, 0): Leaf,
  (1, 1, 0, 1): Leaf,
  (0, 0, 0, 1): Leaf,
  (1, 0, 0, 0): Leaf,
  (0, 1, 1, 1): Leaf,
  (1, 1, 1, 0): Leaf},
 5: {(0, 0, 0, 1, 0): Leaf,
  (1, 0, 0, 1, 1): Leaf,
  (0, 0, 1, 0, 0): Leaf,
  (1, 0, 1, 0, 1): Leaf,
  (0, 1, 0, 0, 0): Leaf,
  (1, 1, 0, 0, 1): Leaf,
  (0, 1, 1, 1, 0): Leaf,
  (1, 1, 1, 1, 1): Leaf,
  (0, 0, 0, 0, 1): Leaf,
  (1, 0, 0, 0, 0): Leaf,
  (0, 0, 1, 1, 1): Leaf,
  (1, 0, 1, 1, 0): Leaf,
  (0, 1, 

In [11]:
ftn_params[0].shape

torch.Size([2, 2, 2])